# Delhi-NCR PM2.5 forecasting — full pipeline on Colab

Runs the whole project end to end: download CPCB station data (via the OpenAQ AWS archive), GFS 1° forecasts and VIIRS 375 m active fires, build the hourly dataset, train every model on both folds and write `reports/RESULTS.md`.

**Runtime → Change runtime type → GPU (T4 is plenty).** The neural models pick the GPU up automatically; downloads take ~45–60 min, training ~30 min on a T4.

In [ ]:
!git clone https://github.com/akhilnathreddy/delhi-aqi-forecast.git
%cd delhi-aqi-forecast
!pip -q install -e ".[download]"
import torch; print("CUDA:", torch.cuda.is_available())

## 1 · Data (all public, no API keys)

In [ ]:
!python -m aqf.cli download-stations   # ~100 stations × 600 station-days
!python -m aqf.cli download-gfs --workers 8
!python -m aqf.cli download-fires --workers 8
!python -m aqf.cli build

Optional: with a NASA FIRMS MAP_KEY you can use the FIRMS API instead of the NOAA granules (same schema).

```python
import os; os.environ['FIRMS_MAP_KEY'] = '...'
!python -m aqf.cli download-fires --source firms
```

## 2 · Train & evaluate

In [ ]:
!python -m aqf.cli train --fold main
!python -m aqf.cli train --fold winter
!python -m aqf.cli report

## 3 · Results

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open('reports/RESULTS.md').read().replace('](figures/', '](reports/figures/')))

In [ ]:
from IPython.display import Image
for f in ['overview', 'map', 'horizon_winter', 'fire_ablation_winter', 'tft_importance_winter', 'example_winter']:
    try: display(Image(f'reports/figures/{f}.png'))
    except FileNotFoundError: pass